In [2]:
import random
from typing import Tuple, Dict

In [3]:
Position = Tuple[int, int]  # (row, col)

# =========================================
# Ambiente 2D (procedurale)
# =========================================

def init_env_2d(rows=4, cols=4, dirt_prob=0.4,
                agent_start_pos: Position = (0, 0),
                seed=None):
    """
    Inizializza un ambiente 2D:
    - rows, cols: dimensioni della griglia
    - dirt_prob: probabilità che ogni cella sia sporca
    - agent_start_pos: posizione iniziale (r, c)
    Restituisce: (rows, cols, dirt, agent_pos, performance)
    """
    if seed is not None:
        random.seed(seed)

    dirt: Dict[Position, bool] = {}
    for r in range(rows):
        for c in range(cols):
            dirt[(r, c)] = (random.random() < dirt_prob)

    agent_pos = agent_start_pos
    performance = 0
    return rows, cols, dirt, agent_pos, performance


def percept_2d(agent_pos: Position, dirt: Dict[Position, bool]):
    """Restituisce ((r, c), è_sporco_qui)."""
    return agent_pos, dirt[agent_pos]


def step_env_2d(rows, cols, dirt, agent_pos, performance, action: str):
    """
    Aggiorna ambiente e posizione in base all'azione.
    Azioni: 'LEFT', 'RIGHT', 'UP', 'DOWN', 'SUCK', 'NOOP'
    Restituisce: (dirt, agent_pos, performance).
    """
    performance -= 1  # costo di passo

    r, c = agent_pos

    if action == "SUCK":
        if dirt[agent_pos]:
            dirt[agent_pos] = False
            performance += 10

    elif action == "LEFT":
        if c > 0:
            c -= 1

    elif action == "RIGHT":
        if c < cols - 1:
            c += 1

    elif action == "UP":
        if r > 0:
            r -= 1

    elif action == "DOWN":
        if r < rows - 1:
            r += 1

    elif action == "NOOP":
        pass

    agent_pos = (r, c)
    return dirt, agent_pos, performance


def all_clean_2d(dirt: Dict[Position, bool]) -> bool:
    """True se tutte le celle sono pulite."""
    return not any(dirt.values())

def show_env_2d(rows, cols, dirt, agent_pos, performance):
    """
    Stampa la griglia con celle a larghezza fissa:
    - " . " per pulito
    - " D " per sporco
    - "(.)" / "(D)" se presente l'agente
    """
    lines = []
    for r in range(rows):
        row_cells = []
        for c in range(cols):
            pos = (r, c)
            dirty_here = dirt[pos]
            if pos == agent_pos:
                ch = "(D)" if dirty_here else "(.)"
            else:
                ch = " D " if dirty_here else " . "
            row_cells.append(ch)
        lines.append("".join(row_cells))
    print("\n".join(lines))
    print(f"(perf={performance})\n")


# =========================================
# Agente 2D con stato esplicito
# =========================================

def init_agent_2d(rows, cols):
    """
    Stato dell'agente:
    - rows, cols: dimensioni
    - (facoltativo) altre info in futuro
    Per ora lo usiamo per una politica di scansione a serpente.
    """
    return {
        "rows": rows,
        "cols": cols,
    }


def agent_2d_policy(agent_state, percept):
    """
    Politica semplice:
    - se la cella è sporca -> 'SUCK'
    - altrimenti scansiona a serpente:
      riga pari: LEFT -> RIGHT
      riga dispari: RIGHT -> LEFT
      alla fine di una riga, scende di una riga.
    Restituisce (azione, nuovo_agent_state).
    """
    rows = agent_state["rows"]
    cols = agent_state["cols"]

    (r, c), is_dirty = percept

    if is_dirty:
        action = "SUCK"
    else:
        # riga pari: ci muoviamo verso destra
        if r % 2 == 0:
            if c < cols - 1:
                action = "RIGHT"
            else:
                # fine riga: se possibile, scendi
                action = "DOWN" if r < rows - 1 else "NOOP"
        # riga dispari: ci muoviamo verso sinistra
        else:
            if c > 0:
                action = "LEFT"
            else:
                # fine riga: se possibile, scendi
                action = "DOWN" if r < rows - 1 else "NOOP"

    # in questo esempio lo stato dell'agente non cambia
    new_agent_state = {
        "rows": rows,
        "cols": cols,
    }
    return action, new_agent_state


# =========================================
# Simulazione episodio 2D
# =========================================

def run_episode_2d(rows=4, cols=4, dirt_prob=0.4,
                   steps=50, agent_start_pos=(0, 0),
                   seed=None):
    rows, cols, dirt, agent_pos, performance = init_env_2d(
        rows=rows,
        cols=cols,
        dirt_prob=dirt_prob,
        agent_start_pos=agent_start_pos,
        seed=seed
    )
    agent_state = init_agent_2d(rows, cols)

    for t in range(steps):
        print(f"t={t:02d}:")
        show_env_2d(rows, cols, dirt, agent_pos, performance)

        percept = percept_2d(agent_pos, dirt)
        action, agent_state = agent_2d_policy(agent_state, percept)

        dirt, agent_pos, performance = step_env_2d(
            rows, cols, dirt, agent_pos, performance, action
        )

        if all_clean_2d(dirt):
            print(f"t={t+1:02d} (tutto pulito):")
            show_env_2d(rows, cols, dirt, agent_pos, performance)
            print("Episodio terminato.")
            break

In [6]:
if __name__ == "__main__":
    run_episode_2d(
        rows=12,
        cols=9,
        dirt_prob=0.1,
        steps=40,
        agent_start_pos=(0, 0),
        seed=123
    )


t=00:
(D) D  .  .  .  D  .  .  . 
 .  .  .  .  D  .  D  .  D 
 .  .  .  D  .  .  D  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  D  .  .  .  .  .  . 
 .  .  D  .  .  .  .  D  . 
 D  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  D  . 
 .  .  .  .  .  .  .  D  . 
 D  .  D  .  D  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
(perf=0)

t=01:
(.) D  .  .  .  D  .  .  . 
 .  .  .  .  D  .  D  .  D 
 .  .  .  D  .  .  D  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  D  .  .  .  .  .  . 
 .  .  D  .  .  .  .  D  . 
 D  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  D  . 
 .  .  .  .  .  .  .  D  . 
 D  .  D  .  D  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
(perf=9)

t=02:
 . (D) .  .  .  D  .  .  . 
 .  .  .  .  D  .  D  .  D 
 .  .  .  D  .  .  D  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  D  .  .  .  .  .  . 
 .  .  D  .  .  .  .  D  . 
 D  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  D  . 
 .  .  .  .  .  .  .  D  . 
 D  .  D  